In [40]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4

max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250
n_embd = 384
n_layer = 4

cuda


In [41]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as file:
    text = file.read()

chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '#', '&', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '\ufeff']


In [42]:
# Tokenizer
# Encoder, decoder

# (map char to its idx)
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]  # encoder: take a string, output a list of integers
decode = lambda l: ''.join(int_to_string[i] for i in l)  # decoder: take a list of integers, output a string

# encoder and decoder because Computers understand numbers, not text

# A tensor is a mathematical object representing data that has multiple dimensions
# generalization of numbers, vectors, and matrices.

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([81, 41, 57, 54,  1, 37, 67, 64, 59, 54, 52, 69,  1, 28, 70, 69, 54, 63,
        51, 54, 67, 56,  1, 54, 23, 64, 64, 60,  1, 64, 55,  1, 41, 57, 54,  1,
        44, 64, 63, 53, 54, 67, 55, 70, 61,  1, 44, 58, 75, 50, 67, 53,  1, 64,
        55,  1, 36, 75,  0,  1,  1,  1,  1,  0, 41, 57, 58, 68,  1, 54, 23, 64,
        64, 60,  1, 58, 68,  1, 55, 64, 67,  1, 69, 57, 54,  1, 70, 68, 54,  1,
        64, 55,  1, 50, 63, 74, 64, 63, 54,  1])


In [43]:
# use 80% of the dataset to train
# 20% for validation (make sure that the prediction is valid)

n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

# get batch to process parallel
# x is inputs, y is targets (predict the next token) 

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,)) # random starting point, length = batch_size
    x = torch.stack([data[i: i+block_size] for i in ix])
    y = torch.stack([data[i+1: i+block_size+1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)

# LLM: Given the previous token, predict the next token
# create (context -> next token) training pairs

# block_size: model can look at maximum 8 previous tokens (context length)
#             block size because transformer cannot look at infinite text
# batch_size: number of training sequences processed parallel
#             speed up training

inputs:
tensor([[74,  1, 55, 64, 70, 63, 53,  1],
        [ 1, 63, 64, 67,  1, 72, 58, 75],
        [64,  1, 51, 54,  1, 68, 70, 67],
        [72, 58, 69, 57,  1, 52, 64, 70]], device='cuda:0')
targets:
tensor([[ 1, 55, 64, 70, 63, 53,  1, 69],
        [63, 64, 67,  1, 72, 58, 75, 50],
        [ 1, 51, 54,  1, 68, 70, 67, 54],
        [58, 69, 57,  1, 52, 64, 70, 63]], device='cuda:0')


In [44]:
# ----------------------- Gradient Descent -----------------------------
#gradient (rate of change) descent (slope)

# Maximization and minimization
# what is the x value, find min or max output of f(x)
# Min or max point has Derivative of 0

# Walk toward that min, max point
# x next = x now - learning_rate * derivative * x now
# learning_rate is very small step

# https://docs.google.com/presentation/d/1nHesO6x48OHMN-XWljFAwnFa1MNCRKtMmj9pEVlNLfs/edit?usp=sharing


# ---------------------- Cross Entropy ------------------------------------
# loss function to measure the difference between two probability distributions
# between true labels (P) and the model's predictions (Q)
# Smaller loss: better predictions

# https://docs.google.com/presentation/d/1ZKqgPiYKPKGgjvbnf03aWKxBtSm4NRmO6qHzlZWkjbA/edit?usp=sharing

In [45]:
# Report loss helper
@torch.no_grad()  # disables gradient tracking because we don't need learning -> faster, less memory
def estimate_loss():
    out = {}
    model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model.forward(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()   # mean of eval_iters loss
        
    model.train()
    return out

# temporarily turn on eval mode and then turn back to train mode for training model later
# eval mode disable dropout (random neurons are turned off to prevent overfitting), BatchNorm, training-specific behavior 
    # (Overfitting is The model memorizes the training data too much instead of learning general patterns)

In [46]:
class BigramLanguageModel(nn.Module):   #nn.Module is the base class for ALL neural networks in PyTorch
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)    
        # number of tokens = vocab_size, embedding dimension = vocab_size
        # current token -> probabilities of next token

    def forward(self, index, targets=None):     # Define the computation performed at every call
        logits = self.token_embedding_table(index)
        # index: input token -> lookup the corresponding vectors in the table  (index.shape = (B, T))
        # logits: raw prediction scores
        # later convert to probabilities using softmax (in cross_entropy)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape          # B: batch_size, T: sequence length, C: vocab size
            logits = logits.view(B*T, C)    
            targets = targets.view(B*T)
            # flatten. cross_entropy expects shape (N, C): N = number of training examples, C = number of classes
            # batch 1 (prediction1, p2) batch2 (p3, p4) -> [p1, p2, p3, p4]
            
            loss = F.cross_entropy(logits, targets)   # computes loss: prediction i vs target i

        return logits, loss


    # generates next tokens
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)         #loss measures how wrong the model was
            # focus only on the last token after current sequence (predicted next token)
            logits = logits[:, -1, :] # becomes (B, C)

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)  # dim=-1 -> apply softmax across classes (C)

            # samples one token from the probability distribution
            index_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append new token to the existing sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)

        return index



model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


“—;—&LHV]h.B‘g&t ’dZH-jSCjEXs/jM3C]l:r#Sy)D*Yo(V2XLU(i EA’-—.‘iyJJL1VEuV2)z5f9U&F(idf5QYObW:;twkdzgz”Ry9Xsk2kN&’9;R“iDIgLj0cpeA-gz:uw4Kkkku)b:fj.R-[w/sbx,Z;9h,w4#w5jCzT?UX4iSNI3]F﻿
ppea/.D[STILA﻿kMy?3CziLD
U.,2HzT-﻿TiSRM;mjr?bLDJAL?zkzdZpB]lGG&
T— ’?gz:R“!VQ]q“0RN4SNZNEUmpZkwUu“k‘Gu& R22
ux’yj/xhhMz!i,iWp/z&fwz2:s1C,?XO2HeXx,‘h!-RA’“wI0htcJkMwBk0’Zw0‘ScTNE/I1](san]CPHa1ikdPZfk((s3eZZo2LBgjkOb;MeN2aTXD
vjx1
*x T—q‘]Tn;Zw]CjvUV2i5i3;lhE9fkpZd)T-K9[JcHV20?r:d15“*.“tACidWACbS!ij”‘/jVNeoK*.aLD—N#nCz*


In [47]:
# Training loop and Optimizer -> better prediction

# create a PyTorch optimizer (AdamW is a popular optimizer for transformer, similar to gradient descent but smarter)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    # Report loss
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # predict then evaluate the loss
    logits, loss = model.forward(xb, yb)

    optimizer.zero_grad(set_to_none=True)  # 1. clear old gradients before loss.backward() (gradient is the slope/derivative)
    loss.backward()                        # 2. compute gradient for each param (backpropagation)
    optimizer.step()                       # 3. update the model parameters (weights) using the gradients (~ w new = w - learning_rate * gradient)

print(loss.item())


# Training Loop
# for many iterations:
#     get batch of data
#     make predictions
#     compute loss (how wrong the model is)
#     compute gradients
#     update weights

step: 0, train loss: 4.805, val loss: 4.824
step: 250, train loss: 4.753, val loss: 4.762
step: 500, train loss: 4.701, val loss: 4.686
step: 750, train loss: 4.642, val loss: 4.642
4.455501556396484


In [48]:
# ------------------ Optimizers ------------------------
# Mean Squared Error (MSE): (loss function)
#     MSE is a common loss function used in regression problems, where the goal is to predict a continuous output. It measures the 
#     average squared difference between the predicted and actual values, and is often used to train neural networks for regression tasks.
# Gradient Descent (GD): 
#     is an optimization algorithm used to minimize the loss function of a machine learning model. 
#     The loss function measures how well the model is able to predict the target variable based on the input features. 
#     The idea of GD is to iteratively adjust the model parameters in the direction of the steepest descent of the loss function
# Momentum: 
#     Momentum is an extension of SGD (Stochastic Gradient Descent) that adds a "momentum" term to the parameter updates. 
#     This term helps smooth out the updates and allows the optimizer to continue moving in the right direction,
#     even if the gradient changes direction or varies in magnitude. 
#     Momentum is particularly useful for training deep neural networks, CNNs.
# RMSprop: 
#     adapt the learning rate of each parameter by using a moving average of the squared gradient. 
#     This helps to avoid oscillations in the parameter updates and can improve convergence in some cases.
# Adam: 
#     Adam is a popular optimization algorithm that combines the ideas of momentum and RMSprop. It uses a moving average of 
#     both the gradient and its squared value to adapt the learning rate of each parameter. Adam is often used as a default optimizer for deep learning models.
# AdamW: 
#     AdamW is a modification of the Adam optimizer that adds weight decay to the parameter updates. This helps to regularize the model and 
#     can improve generalization performance. We will be using the AdamW optimizer as it best suits the properties of the model we will train in this video.
# find more optimizers and details at torch.optim

In [49]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


b.F﻿pWp4NXctov!Ngz‘5ngo2o/sbc#Ek”c[b“fc﻿pp/pp[trgEN9Qd1u—UCTnPRb9sElG2iJL#P2Z(G*vt “w,vyxFSy,g]yvf&1D]wUkp‘)&AH4#Q]T4K‘(jn5cYmh.,niD
V29Z4#.lWT“9]2KBNQjlT—h”WbJ.G”3;F&NXj#JZx(h’?XsxF1﻿—ZWvSqKpfE*V2L’?﻿k3b.;R&yi1-Kk#K&
1T/#Wfkppf&‘-owD—;[wkdzU.,/jg3ru*0vonFMo[
*(5LJKC,?/jrIfC5Sfr-/ &ZWIPjaY‘IidHBAJV“1H2?c,2EAYdQ V]ChMwkMzg‘”g#avBAMx avr:; 3BTSGq‘Yms1(-BfidkLD#aJHGeoZ//A1)&FLj#ZU.;npNg”.yEXV]qOsBO(j(zf)uo:9“*)”9cB/tBxjhg1xF[xFLX:;hWA-v—y3HnMT-[g* 34!n&dst R—ZUX.,BupS-Bbi‘?t2z‘“—zWCYCPbCD]V2:bO/


In [ ]:
# ---------------- Normalization ---------------------------
# Rescaling data into a more stable and consistent range so models can learn better
    
# | Type                  | Used For                          |
# | --------------------- | --------------------------------- |
# | Input normalization   | Scale raw data                    |
# | Standardization       | Center around mean                |
# | BatchNorm             | Normalize hidden activations      |
# | LayerNorm             | Normalize Transformer activations |
# | Min-Max normalization | Scale into fixed range            |


In [ ]:
# Activation functions: ReLU, Sigmoid, tanh
# introduces nonlinearity -> network can approximate extremely complex functions

# ReLU(x) = max(0,x)    better

# Sigmoid: S-shape, out range [0, 1]
# tanh: S-shape, out range [-1, 1]
# zero-centered outputs, better than sigmoid
# both Sigmoid and tanh causes vanishing gradients (large input -> out is either -1 or 1]

In [ ]:
# --------------- Transformer Architecture --------------------------
# Attention is all you need (https://arxiv.org/pdf/1706.03762)